In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

liste_competences = [
    "solidworks", "catia", "autocad", "creo", "nx", "pro/engineer","inventor", 
    "ansys", "matlab", "simulink", "abaqus", "samcef","nastran", "patran", "icem", 
    "cfd", "fea", "rdm", "cao", "dao", "conception mécanique", "modélisation",
    "fabrication additive", "gmao", "sap", "erp", "ms project", "automatisme", 
    "robotique", "amdec", "apqp", "ppap", "spc", "5s", "kaizen", "lean", 
    "lean manufacturing", "six sigma", "python", "sql", "vba", "excel"
]
liste_soft_skills = [
    "communication", "communiquer", "esprit d'équipe", "travail en équipe", "travailler en équipe",
    "autonomie","autonome", "rigueur", "organisation", "organisé", "créativité", "créatif",
    "leadership", "gestion de projet","innovation", "résolution de problèmes", "adaptabilité",
    "force de proposition", "proactivité", "relationnel", "initiative", "polyvalence", "flexibilité", 
    "sens de l'organisation", "réactivité", "esprit d'analyse"
]
liste_secteurs = [
    "automobile", "aéronautique", "métallurgie", "btp", "construction",
    "énergie", "agroalimentaire", "ferroviaire", "naval", "chimie",
    "plasturgie", "minier", "logistique", "bureau d'études",
    "pharmaceutique", "procédés", "recherche scientifique"
]
liste_contrats = ["cdi", "cdd", "stage", "alternance", "intérim", "anapec"]

# Lecture du fichier des villes marocaines
fichier_villes = open("villes_maroc.txt", "r", encoding="utf-8")
lignes = fichier_villes.readlines()
fichier_villes.close()

# Transformation du fichier en liste de villes marocaines
villes = []
for ligne in lignes:
    ligne = ligne.strip()
    if ligne != "":
        villes.append(ligne)
villes.sort(key=len, reverse=True)

liste_offres = []
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}  # adresse pour ne pas etre bloque par le site

#Iteration sur chaque page(et son propre url)
for page in range(1, 68):
    offre_page = 0
    url = "https://www.dreamjob.ma/page/" + str(page) + "/?s=" + "genie+mecanique"
    print(f"Scraping de la page {page}:")
    try: 
        response = requests.get(url, headers=headers, timeout=15)         # envoie de requetes au site
    except Exception as e: 
        print(f"Erreur de connexion à la page {page}: {e}")
        continue

    if response.status_code != 200:
        print(f"Erreur, page {page} (Code: {response.status_code})")
        continue

    soup = BeautifulSoup(response.text, "html.parser")                    # lecture et analyse du code HTML
    articles = soup.select("article.jeg_post")

    for art in articles:                                                   # parcours de chaque offre
        titre_tag = art.select_one("h3.jeg_post_title a")
        if titre_tag is None:
            continue

        titre = titre_tag.text.strip()                                    # extraction du titre et du lien
        lien = titre_tag.get("href", "")

        extrait_tag = art.select_one(".jeg_post_excerpt p")
        extrait = extrait_tag.text.strip() if extrait_tag is not None else ""

        texte_total = (titre + " " + extrait).lower()
        ville_trouvee = "Non spécifié"
        for v in villes:
            if v.lower() in texte_total:
                ville_trouvee = v
                break

        resultat = re.search(r'^(.*?)\s+(recrute|cherche|ouvre)\b', titre, re.IGNORECASE)
        entreprise = resultat.group(1).strip() if resultat is not None else "Non spécifié"

    
        # Extraction des details de chaque offre
        if lien != "":
            try:
                time.sleep(1)
                resp_offre = requests.get(lien, headers=headers, timeout=15)
                
                if resp_offre.status_code == 200:
                    soup_offre = BeautifulSoup(resp_offre.text, "html.parser")
                    contenu_offre = soup_offre.select_one(".entry-content")
                    
                    if contenu_offre is not None:
                        texte_offre = contenu_offre.text.lower()
                        texte_recherche_complet = (titre + " " + contenu_offre.text).lower()
                        
                        # Langues
                        langues = []
                        for l in ["français", "francais", "anglais", "english", "arabe"]:                    #recherche des langues dans le texte
                            if l in texte_offre:
                                langues.append(l.capitalize())
                        langue = ", ".join(langues) if langues else "Non specifié"
                        
                        contrat = "Non spécifié"
                        niveau_etude = "Non spécifié"
                        experience = "Non spécifié"
                        secteur = "Non spécifié"
                        competences_techniques = "Non spécifié"
                        soft_skills = "Non spécifié"
                        
                        # Contrat proposé
                        contrat_trouve = []
                        for c in liste_contrats:
                            if c in texte_offre:
                                if c in ("cdi", "cdd"):
                                    contrat = c.upper()
                                else:
                                    contrat = c.capitalize()
                                break
                        
                        # Niveau d'étude
                        match_etude = re.findall(r'bac\s*\+\s*\d+', texte_offre)
                        if match_etude :
                            niveau_etude = ", ".join(
                                    sorted(set(e.replace(" ", "") for e in match_etude))).capitalize()
                        
                        # Expérience
                        match_exp = re.search(r'(\d+\s*(?:à|-)\s*\d+\s*ans|\d+\s*ans|\d+\s*an)', texte_offre)
                        if match_exp is not None:
                            experience = match_exp.group(1) 
                        
                        # Secteur d'activité
                        secteurs_trouves = []
                        for sec in liste_secteurs:
                            if re.search(r'\b' + re.escape(sec) + r'\b', texte_recherche_complet):
                                formatted_sec = sec.replace(" d\\'", " d'").capitalize()
                                secteurs_trouves.append(formatted_sec)
                        if secteurs_trouves:
                            secteur = ", ".join(sorted(list(set(secteurs_trouves))))

                        # Compétences Techniques
                        ct_trouvees = []
                        for comp in liste_competences:
                            if re.search(r'\b' + re.escape(comp) + r'\b', texte_recherche_complet):
                                if comp in ["solidworks", "catia", "nx", "ansys", 
                                            "matlab", "samcef", "nastran", "icem", 
                                            "cfd", "fea", "rdm", "cao", "dao", 
                                            "gmao", "sap", "erp", "amdec", "apqp", 
                                            "ppap", "spc", "5s", "sql", "vba"]:   
                                    ct_trouvees.append(comp.upper())
                                else :
                                    ct_trouvees.append(comp.capitalize())
                        if ct_trouvees:       
                            competences_techniques = ", ".join(sorted(list(set(ct_trouvees)))) 

                        # Soft Skills
                        ss_trouves = []
                        for skill in liste_soft_skills:
                            if re.search(r'\b' + re.escape(skill) + r'\b', texte_recherche_complet):
                                formatted_skill = skill.replace(" d\'", " d'").replace("communiquer", "communication").capitalize()
                                ss_trouves.append(formatted_skill)
                        if ss_trouves:
                            soft_skills = ", ".join(sorted(list(set(ss_trouves)))) 

            except Exception as e:
                print(f"Erreur lors du traitement de l'URL {lien}: {e}")
        
        
        offre = {
            "Titre du poste": titre, 
            "Entreprise": entreprise, 
            "Ville": ville_trouvee, 
            "Secteur": secteur,
            "Contrat proposé": contrat, 
            "Niveau d'étude": niveau_etude, 
            "Expérience": experience,  
            "Compétences Techniques": competences_techniques, 
            "Langues": langue,
            "Soft Skills": soft_skills, 
            "Lien de l'offre": lien
        }
        liste_offres.append(offre)
        offre_page+=1
    print(f"{offre_page} offres")

    time.sleep(1.5)

# Exportation des resultats vers un fichier CSV
if len(liste_offres) > 0:
    df_dreamjob = pd.DataFrame(liste_offres)
    df_dreamjob.to_csv("offres_dreamjob_mecanique.csv", index=False, sep=",", encoding="utf-8-sig")
    print(f"\nExtraction terminée. Total : {len(df_dreamjob)} offres.")
else:
    print("Aucune offre n'a été trouvée.")



Scraping de la page 1:
31 offres
Scraping de la page 2:
31 offres
Scraping de la page 3:
31 offres
Scraping de la page 4:
31 offres
Scraping de la page 5:
31 offres
Scraping de la page 6:
31 offres
Scraping de la page 7:
31 offres
Scraping de la page 8:
31 offres
Scraping de la page 9:
31 offres
Scraping de la page 10:
31 offres
Scraping de la page 11:
31 offres
Scraping de la page 12:
31 offres
Scraping de la page 13:
31 offres
Scraping de la page 14:
31 offres
Scraping de la page 15:
31 offres
Scraping de la page 16:
31 offres
Scraping de la page 17:
31 offres
Scraping de la page 18:
31 offres
Scraping de la page 19:
31 offres
Scraping de la page 20:
31 offres
Scraping de la page 21:
31 offres
Scraping de la page 22:
31 offres
Scraping de la page 23:
31 offres
Scraping de la page 24:
31 offres
Scraping de la page 25:
31 offres
Scraping de la page 26:
31 offres
Scraping de la page 27:
31 offres
Scraping de la page 28:
31 offres
Scraping de la page 29:
31 offres
Scraping de la page 30:

In [2]:
import pandas as pd
# On charge en lisant avec une virgule (sep=",")
df = pd.read_csv("offres_dreamjob_mecanique.csv", sep=",")
df.head(20)

,Titre du poste,Entreprise,Ville,Secteur,Contrat proposé,Niveau d'étude,Expérience,Compétences Techniques,Langues,Soft Skills,Lien de l'offre
0,SIDEN cherche 4 ingénieurs en génie mécanique,SIDEN,Jorf Lasfar,Construction,CDI,Non spécifié,Non spécifié,Non spécifié,Non specifié,"Communication, Rigueur",https://www.dreamjob.ma/emploi/siden-cherche-4...
1,L’Université Internationale de Casablanca recr...,L’Université Internationale de Casablanca,Casablanca,Recherche scientifique,Non spécifié,Non spécifié,Non spécifié,Non spécifié,Non specifié,Innovation,https://www.dreamjob.ma/emploi/luniversite-int...
2,SIDEN Recrute: Ingénieurs en Construction Méta...,SIDEN,Non spécifié,Construction,Non spécifié,Non spécifié,1 an,Non spécifié,Non specifié,"Créativité, Rigueur",https://www.dreamjob.ma/emploi/siden-recrute-i...
3,SCIF recrute des Ingénieurs Génie Mécanique,SCIF,Casablanca,Ferroviaire,Non spécifié,Non spécifié,Non spécifié,"CAO, Lean",Non specifié,"Communication, Gestion de projet, Résolution d...",https://www.dreamjob.ma/emploi/scif-recrute-de...
4,Capgemini Engineering recrute 30 techniciens e...,Capgemini Engineering,Casablanca,"Automobile, Aéronautique, Ferroviaire, Plastur...",Non spécifié,Bac+2,50 ans,"CATIA, NX","Français, Anglais",Organisation,https://www.dreamjob.ma/emploi/capgemini-engin...
5,Chantiers et Ateliers du Maroc recrute des Tec...,Chantiers et Ateliers du Maroc,Casablanca,Naval,Non spécifié,Non spécifié,1 an,"Autocad, Ms project",Non specifié,Non spécifié,https://www.dreamjob.ma/emploi/chantiers-et-at...
6,"CEC ouvre des postes en génie civil, mécanique...",CEC,Non spécifié,"Construction, Minier",Non spécifié,Non spécifié,6 ans,Non spécifié,Non specifié,"Leadership, Rigueur",https://www.dreamjob.ma/emploi/cec-ouvre-des-p...
7,Dama recrute un Ingénieur en Maintenance et As...,Dama,Casablanca,Non spécifié,CDI,Non spécifié,30 ans,Non spécifié,Anglais,Communication,https://www.dreamjob.ma/emploi/dama-recrute-un...
8,Safran Recrute des Ingénieurs en Mécanique et ...,Safran,Casablanca,"Automobile, Aéronautique",Stage,Non spécifié,Non spécifié,"ANSYS, Abaqus, Autocad, CATIA, ICEM, NASTRAN, ...",Non specifié,Communication,https://www.dreamjob.ma/emploi/safran-recrute-...
9,Safran recherche activement des Ingénieurs Méc...,Non spécifié,Casablanca,"Automobile, Aéronautique",Stage,Non spécifié,Non spécifié,"ANSYS, Abaqus, Autocad, CATIA, ICEM, NASTRAN, ...",Non specifié,Communication,https://www.dreamjob.ma/emploi/safran-recherch...
